# ⚠️ NOTE
This notebook was evaluated on the original SemEval dev set which had 2,651 tweets 
overlapping with train — results are inflated and NOT valid.

**Corrected LR baseline (clean 85/15 split): Accuracy=0.587, Macro F1=0.592**

See 03-mbert.ipynb for correct evaluation.

In [1]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [5]:
import sys
sys.path.append("..")

from src.data_loader import load_conll

In [6]:
train_df = load_conll("../data/hinglish_train.txt")
dev_df = load_conll("../data/hinglish_dev.txt")

In [7]:
print(train_df.shape)
print(dev_df.shape)

train_df.head()

(15131, 3)
(3000, 3)


,tweet_id,sentiment,text
0,3,negative,@ AdilNisarButt pakistan ka ghra tauq he Pakis...
1,41,negative,Madarchod mulle ye mathura me Nahi dikha tha j...
2,48,positive,@ narendramodi Manya Pradhan Mantri mahoday Sh...
3,64,positive,@ Atheist _ Krishna Jcb full trend me chal rah...
4,66,positive,@ AbhisharSharma _ @ RavishKumarBlog Loksabha ...


In [8]:
X_train = train_df["text"]
y_train = train_df["sentiment"]

X_dev = dev_df["text"]
y_dev = dev_df["sentiment"]

print(X_train.shape)
print(y_train.shape)

(15131,)
(15131,)


In [9]:
#vectorize
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_dev_tfidf = vectorizer.transform(X_dev)

print(X_train_tfidf.shape)
print(X_dev_tfidf.shape)

(15131, 10000)
(3000, 10000)


In [ ]:
#training model (TF-IDF Logistic Regression)
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Training complete!")

Training complete!


In [ ]:
#prediction
preds = model.predict(X_dev_tfidf)

print(preds[:10])

['positive' 'negative' 'negative' 'neutral' 'positive' 'positive'
 'positive' 'positive' 'positive' 'positive']


In [12]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:",
      round(accuracy_score(y_dev, preds), 4))

print("\nMacro F1:",
      round(f1_score(y_dev, preds,
                     average="macro"), 4))

print("\nClassification Report:\n")
print(classification_report(y_dev, preds))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_dev, preds))

Accuracy: 0.7657

Macro F1: 0.7674

Classification Report:

              precision    recall  f1-score   support

    negative       0.77      0.80      0.78       890
     neutral       0.75      0.73      0.74      1128
    positive       0.78      0.78      0.78       982

    accuracy                           0.77      3000
   macro avg       0.77      0.77      0.77      3000
weighted avg       0.77      0.77      0.77      3000


Confusion Matrix:

[[709 135  46]
 [141 822 165]
 [ 71 145 766]]


In [13]:
feature_names = vectorizer.get_feature_names_out()

print("Vocabulary size:",
      len(feature_names))

Vocabulary size: 10000


In [14]:
feature_names = vectorizer.get_feature_names_out()

In [20]:
print(model.coef_.shape)

print(model.classes_)

(3, 10000)
['negative' 'neutral' 'positive']


In [21]:
import pandas as pd
import numpy as np

for i, sentiment in enumerate(model.classes_):
    
    top_idx = np.argsort(model.coef_[i])[-20:]

    top_words = pd.DataFrame({
        "word": [feature_names[idx] for idx in reversed(top_idx)],
        "weight": [round(model.coef_[i][idx], 3) for idx in reversed(top_idx)]
    })

    print(f"\n===== {sentiment.upper()} =====")
    display(top_words)


===== NEGATIVE =====


,word,weight
0,bc,3.736
1,chutiya,3.432
2,chutiye,3.368
3,sale,3.086
4,fuck,3.050
5,chor,3.019
6,kutte,2.571
7,saale,2.550
8,shame,2.445
9,sala,2.349



===== NEUTRAL =====


,word,weight
0,follow,1.535
1,tou,1.334
2,20,1.215
3,maybe,1.111
4,tweet,1.096
5,khilaf,1.036
6,we have,1.003
7,no,0.999
8,uss,0.994
9,wo https,0.990



===== POSITIVE =====


,word,weight
0,love,2.934
1,congratulations,2.791
2,good,2.511
3,happy,2.481
4,great,2.395
5,allah,2.386
6,best,2.290
7,proud,2.227
8,thank,2.165
9,mubarak,2.157
